In [0]:
%scala

import org.apache.spark.sql.functions._

// Read Bronze customers
val customers = spark.table("retail.bronze.customers")

println("========== CUSTOMER DATA SCHEMA ==========")
customers.printSchema()

// 1. Total records
println("========== TOTAL RECORDS ==========")
println(s"Total customers: ${customers.count()}")

// 2. Display all records
println("========== ALL CUSTOMERS ==========")
customers.show(50, false)

// 3. Duplicate customer IDs
println("========== DUPLICATE CUSTOMER IDS ==========")
customers
  .groupBy("customer_id")
  .count()
  .filter(col("count") > 1)
  .show(false)

// 4. Missing customer details
println("========== MISSING CUSTOMER DETAILS ==========")

customers.filter(
  col("customer_id").isNull || trim(col("customer_id")) === "" ||
  col("customer_name").isNull || trim(col("customer_name")) === "" ||
  col("email").isNull || trim(col("email")) === "" ||
  col("city").isNull || trim(col("city")) === "" ||
  col("signup_date").isNull || trim(col("signup_date")) === ""
).show(false)

// 5. Invalid email formats
println("========== INVALID EMAILS ==========")

customers.filter(
  !lower(trim(col("email"))).rlike(
    "^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$"
  )
).show(false)

// 6. Invalid or future signup dates
println("========== INVALID SIGNUP DATES ==========")

val customersWithDate = customers.withColumn(
  "_signup_date",
  expr("try_cast(signup_date AS DATE)")
)

customersWithDate.filter(
  col("_signup_date").isNull ||
  col("_signup_date") > current_date()
).show(false)

// 7. City casing issues
println("========== CITY STANDARDIZATION ISSUES ==========")

customers.filter(
  col("city") =!= initcap(trim(col("city")))
).show(false)

println("========== CUSTOMER PROFILING COMPLETED ==========")


========== CUSTOMER DATA SCHEMA ==========
root
-- customer_id: string (nullable = true)
-- customer_name: string (nullable = true)
-- email: string (nullable = true)
-- city: string (nullable = true)
-- signup_date: string (nullable = true)
-- _ingestion_timestamp: timestamp (nullable = true)
-- _source_file: string (nullable = true)
-- _batch_id: string (nullable = true)

========== TOTAL RECORDS ==========
Total customers: 42
========== ALL CUSTOMERS ==========
+-----------+--------------+------------------------------+---------+-----------+--------------------------+----------------------------------------------------------------+---------+
customer_id|customer_name |email |city |signup_date|_ingestion_timestamp |_source_file |_batch_id|
+-----------+--------------+------------------------------+---------+-----------+--------------------------+----------------------------------------------------------------+---------+
CUST001 |Priya Patel |priya.patel1@mailexample.in |Mumbai |2025-06-24 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST002 |Sneha Gupta |sneha.gupta2@mailexample.in |Kolkata |2025-03-13 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST003 |Divya Singh |divya.singh3@mailexample.in |Kochi |2025-02-08 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST004 |Suresh Gupta |suresh.gupta4@mailexample.in |Kochi |2023-09-03 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST005 |Aarav Gupta |aarav.gupta5@mailexample.in |Mumbai |2023-04-24 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST006 |Vikram Sharma |vikram.sharma6@mailexample.in |Kochi |2023-01-12 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST007 |Lakshmi Rao |lakshmi.rao7@mailexample.in |Pune |2023-08-04 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST008 |Karthik Khan |invalid.email.example.in |Pune |2025-08-09 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST009 |Anita Das |anita.das9@mailexample.in |Mumbai |2025-01-10 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST010 |Anita Sharma |anita.sharma10@mailexample.in |hyderabad|2023-10-22 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST011 |Meera Iyer |meera.iyer11@mailexample.in |Chennai |2023-01-26 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST012 |Lakshmi Patel |lakshmi.patel12@mailexample.in|Pune |2024-04-17 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST013 |Karthik Sharma|NULL |Kolkata |2025-07-01 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST014 |Divya Singh |divya.singh14@mailexample.in |Kochi |2025-06-07 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST015 |Priya Das |priya.das15@mailexample.in |MUMBAI |2024-02-12 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST016 |Rahul Singh |rahul.singh16@mailexample.in |Chennai |2025-05-14 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST017 |NULL |karthik.gupta17@mailexample.in|Kochi |2024-06-09 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST018 |Imran Reddy |imran.reddy18@mailexample.in |Bengaluru|2025-02-23 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.cor

In [0]:
%scala

// Business columns used to compare customer records
val customerColumns = Seq(
  "customer_id",
  "customer_name",
  "email",
  "city",
  "signup_date"
)

// Analyze duplicate customer IDs
val duplicateCustomerAnalysis = customers
  .groupBy("customer_id")
  .agg(
    count("*").alias("total_count"),
    countDistinct(
      struct(customerColumns.map(col): _*)
    ).alias("distinct_versions")
  )
  .filter(col("total_count") > 1)

// Exact duplicates
val exactDuplicateCustomerIds = duplicateCustomerAnalysis
  .filter(col("distinct_versions") === 1)
  .select("customer_id")

// Conflicting duplicates
val conflictingCustomerIds = duplicateCustomerAnalysis
  .filter(col("distinct_versions") > 1)
  .select("customer_id")

println("========== CUSTOMER DUPLICATE ANALYSIS ==========")

println("Exact duplicate customer IDs:")
exactDuplicateCustomerIds.show(false)

println("Conflicting duplicate customer IDs:")
conflictingCustomerIds.show(false)


========== CUSTOMER DUPLICATE ANALYSIS ==========
Exact duplicate customer IDs:
+-----------+
customer_id|
+-----------+
CUST005 |
CUST020 |
+-----------+

Conflicting duplicate customer IDs:
+-----------+
customer_id|
+-----------+
+-----------+

val customerColumns: Seq[String] = List(customer_id, customer_name, email, city, signup_date)
val duplicateCustomerAnalysis: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [customer_id: string, total_count: bigint ... 1 more field]
val exactDuplicateCustomerIds: org.apache.spark.sql.DataFrame = [customer_id: string]
val conflictingCustomerIds: org.apache.spark.sql.DataFrame = [customer_id: string]

In [0]:
%scala
// Remove exact duplicate customer records
val deduplicatedCustomers = customers.dropDuplicates(customerColumns)

// Identify any conflicting customer records
val conflictingCustomers = customers.join(
  conflictingCustomerIds,
  Seq("customer_id"),
  "inner"
)

// Exclude conflicting customer IDs from further processing
val customersAfterDuplicateHandling = deduplicatedCustomers.join(
  conflictingCustomerIds,
  Seq("customer_id"),
  "left_anti"
)

println(s"Original customers: ${customers.count()}")
println(s"After exact deduplication: ${deduplicatedCustomers.count()}")
println(s"Conflicting customer records: ${conflictingCustomers.count()}")
println(s"After duplicate handling: ${customersAfterDuplicateHandling.count()}")

Original customers: 42
After exact deduplication: 40
Conflicting customer records: 0
After duplicate handling: 40
val deduplicatedCustomers: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [customer_id: string, customer_name: string ... 6 more fields]
val conflictingCustomers: org.apache.spark.sql.DataFrame = [customer_id: string, customer_name: string ... 6 more fields]
val customersAfterDuplicateHandling: org.apache.spark.sql.DataFrame = [customer_id: string, customer_name: string ... 6 more fields]

In [0]:
%scala

val standardizedCustomers = customersAfterDuplicateHandling
  .withColumn("customer_name", trim(col("customer_name")))
  .withColumn("email", lower(trim(col("email"))))
  .withColumn("city", initcap(trim(col("city"))))
  .withColumn(
    "signup_date",
    coalesce(
      expr("try_to_date(trim(signup_date), 'yyyy-MM-dd')"),
      expr("try_to_date(trim(signup_date), 'dd/MM/yyyy')")
    )
  )

// Display standardized customer records
standardizedCustomers.select(
  "customer_id",
  "customer_name",
  "email",
  "city",
  "signup_date"
).show(50, false)

// Verify updated schema
standardizedCustomers.printSchema()

+-----------+--------------+------------------------------+---------+-----------+
customer_id|customer_name |email |city |signup_date|
+-----------+--------------+------------------------------+---------+-----------+
CUST002 |Sneha Gupta |sneha.gupta2@mailexample.in |Kolkata |2025-03-13 |
CUST003 |Divya Singh |divya.singh3@mailexample.in |Kochi |2025-02-08 |
CUST017 |NULL |karthik.gupta17@mailexample.in|Kochi |2024-06-09 |
CUST022 |Neha Rao |neha.rao22@mailexample.in |Kochi |2024-03-15 |
CUST023 |Rahul Singh |rahul.singh23@mailexample.in |Chennai |2024-07-27 |
CUST024 |Karthik Khan |karthik.khan24@mailexample.in |Kochi |2025-04-04 |
CUST009 |Anita Das |anita.das9@mailexample.in |Mumbai |2025-01-10 |
CUST030 |Imran Rao |imran.rao30@mailexample.in |Kolkata |2024-07-25 |
CUST006 |Vikram Sharma |vikram.sharma6@mailexample.in |Kochi |2023-01-12 |
CUST010 |Anita Sharma |anita.sharma10@mailexample.in |Hyderabad|2023-10-22 |
CUST025 |Rahul Das |rahul.das25@mailexample.in |Kolkata |2023-07-27 |
CUST028 |Imran Rao |imran.rao28@mailexample.in |Pune |2025-08-01 |
CUST039 |Meera Gupta |meera.gupta39@mailexample.in |Mumbai |2024-09-16 |
CUST001 |Priya Patel |priya.patel1@mailexample.in |Mumbai |2025-06-24 |
CUST011 |Meera Iyer |meera.iyer11@mailexample.in |Chennai |2023-01-26 |
CUST013 |Karthik Sharma|NULL |Kolkata |2025-07-01 |
CUST014 |Divya Singh |divya.singh14@mailexample.in |Kochi |2025-06-07 |
CUST020 |Divya Das |divya.das20@mailexample.in |Kochi |2023-05-26 |
CUST021 |Meera Nair |meera.nair21@mailexample.in |Mumbai |2024-10-03 |
CUST026 |Vikram Khan |no@domain |Hyderabad|2023-02-03 |
CUST032 |Meera Sharma |meera.sharma32@mailexample.in |Kochi |2024-12-06 |
CUST035 |Neha Rao |neha.rao35@mailexample.in |Mumbai |2024-10-26 |
CUST004 |Suresh Gupta |suresh.gupta4@mailexample.in |Kochi |2023-09-03 |
CUST007 |Lakshmi Rao |lakshmi.rao7@mailexample.in |Pune |2023-08-04 |
CUST018 |Imran Reddy |imran.reddy18@mailexample.in |Bengaluru|2025-02-23 |
CUST034 |Rahul Iyer |rahul.iyer34@mailexample.in |Kolkata |2027-06-01 |
CUST037 |Sneha Das |sneha.das37@mailexample.in |Delhi |2024-04-05 |
CUST038 |Rahul Reddy |rahul.reddy38@mailexample.in |Mumbai |2023-01-01 |
CUST027 |Rohan Gupta |rohan.gupta27@mailexample.in |Kochi |2025-04-02 |
CUST029 |Sneha Khan |sneha.khan29@mailexample.in |Kochi |2024-04-22 |
CUST033 |Neha Singh |neha.singh33@mailexample.in |Chennai |2024-04-08 |
CUST036 |Vikram Rao |vikram.rao36@mailexample.in |Pune |2023-04-09 |
CUST040 |Suresh Nair |suresh.nair40@mailexample.in |Bengaluru|2023-05-04 |
CUST008 |Karthik Khan |invalid.email.example.in |Pune |2025-08-09 |
CUST012 |Lakshmi Patel |lakshmi.patel12@mailexample.in|Pune |2024-04-17 |
CUST015 |Priya Das |priya.das15@mailexample.in |Mumbai |2024-02-12 |
CUST031 |Arjun Reddy |arjun.reddy31@mailexample.in |Delhi |2023-11-21 |
CUST005 |Aarav Gupta |aarav.gupta5@mailexample.in |Mumbai |2023-04-24 |
CUST016 |Rahul Singh |rahul.singh16@mailexample.in |Chennai |2025-05-14 |
CUST019 |Rohan Reddy |rohan.reddy19@mailexample.in |Mumbai |2025-01-04 |
+-----------+--------------+------------------------------+---------+-----------+

root
-- customer_id: string (nullable = true)
-- customer_name: string (nullable = true)
-- email: string (nullable = true)
-- city: string (nullable = true)
-- signup_date: date (nullable = true)
-- _ingestion_timestamp: timestamp (nullable = true)
-- _source_file: string (nullable = true)
-- _batch_id: string (nullable = true)

val standardizedCustomers: org.apache.spark.sql.DataFrame = [customer_id: string, customer_name: string ... 6 more fields]

In [0]:

%scala

val emailRegex = "^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$"

val customersWithReasons = standardizedCustomers.withColumn(
  "rejection_reason",
  concat_ws(
    ", ",
    when(
      col("customer_id").isNull || trim(col("customer_id")) === "",
      lit("MISSING_CUSTOMER_ID")
    ),
    when(
      col("customer_name").isNull || trim(col("customer_name")) === "",
      lit("MISSING_CUSTOMER_NAME")
    ),
    when(
      col("email").isNull || trim(col("email")) === "",
      lit("MISSING_EMAIL")
    ),
    when(
      col("email").isNotNull && !col("email").rlike(emailRegex),
      lit("INVALID_EMAIL")
    ),
    when(
      col("city").isNull || trim(col("city")) === "",
      lit("MISSING_CITY")
    ),
    when(
      col("signup_date").isNull,
      lit("INVALID_OR_MISSING_SIGNUP_DATE")
    ),
    when(
      col("signup_date") > current_date(),
      lit("FUTURE_SIGNUP_DATE")
    )
  )
)

// Separate valid and rejected records
val validCustomers = customersWithReasons.filter(col("rejection_reason") === "")

val rejectedCustomers = customersWithReasons.filter(col("rejection_reason") =!= "")

println(s"Valid customers: ${validCustomers.count()}")
println(s"Rejected customers: ${rejectedCustomers.count()}")

println("========== REJECTED CUSTOMERS ==========")
rejectedCustomers.select(
  "customer_id",
  "customer_name",
  "email",
  "city",
  "signup_date",
  "rejection_reason"
).show(false)

println("========== REJECTION SUMMARY ==========")
rejectedCustomers.groupBy("rejection_reason").count().show(false)

Valid customers: 35
Rejected customers: 5
========== REJECTED CUSTOMERS ==========
+-----------+--------------+------------------------------+---------+-----------+---------------------+
customer_id|customer_name |email |city |signup_date|rejection_reason |
+-----------+--------------+------------------------------+---------+-----------+---------------------+
CUST017 |NULL |karthik.gupta17@mailexample.in|Kochi |2024-06-09 |MISSING_CUSTOMER_NAME|
CUST013 |Karthik Sharma|NULL |Kolkata |2025-07-01 |MISSING_EMAIL |
CUST026 |Vikram Khan |no@domain |Hyderabad|2023-02-03 |INVALID_EMAIL |
CUST034 |Rahul Iyer |rahul.iyer34@mailexample.in |Kolkata |2027-06-01 |FUTURE_SIGNUP_DATE |
CUST008 |Karthik Khan |invalid.email.example.in |Pune |2025-08-09 |INVALID_EMAIL |
+-----------+--------------+------------------------------+---------+-----------+---------------------+

========== REJECTION SUMMARY ==========
+---------------------+-----+
rejection_reason |count|
+---------------------+-----+
FUTURE_SIGNUP_DATE |1 |
MISSING_CUSTOMER_NAME|1 |
INVALID_EMAIL |2 |
MISSING_EMAIL |1 |
+---------------------+-----+

val emailRegex: String = ^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$
val customersWithReasons: org.apache.spark.sql.DataFrame = [customer_id: string, customer_name: string ... 7 more fields]
val validCustomers: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [customer_id: string, customer_name: string ... 7 more fields]
val rejectedCustomers: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [customer_id: string, customer_name: string ... 7 more fields]

In [0]:
%scala

// Save clean customers
validCustomers
  .drop("rejection_reason")
  .write
  .format("delta")
  .mode("overwrite")
  .saveAsTable("retail.silver.customers")

// Save rejected customers along with rejection reasons
rejectedCustomers
  .write
  .format("delta")
  .mode("overwrite")
  .saveAsTable("retail.silver.rejected_customers")

println("Customer Silver tables created successfully!")

Customer Silver tables created successfully!

In [0]:
%scala

val silverCustomers = spark.table("retail.silver.customers")
val rejectedCustomersTable = spark.table("retail.silver.rejected_customers")

println(s"Clean customers: ${silverCustomers.count()}")
println(s"Rejected customers: ${rejectedCustomersTable.count()}")

// Check duplicate customer IDs
silverCustomers
  .groupBy("customer_id")
  .count()
  .filter(col("count") > 1)
  .show(false)

// Check missing required fields
silverCustomers.filter(
  col("customer_id").isNull ||
  col("customer_name").isNull ||
  col("email").isNull ||
  col("city").isNull ||
  col("signup_date").isNull
).show(false)

// View clean customers
silverCustomers.show(50, false)

// Rejection summary
rejectedCustomersTable
  .groupBy("rejection_reason")
  .count()
  .show(false)

Clean customers: 35
Rejected customers: 5
+-----------+-----+
customer_id|count|
+-----------+-----+
+-----------+-----+

+-----------+-------------+-----+----+-----------+--------------------+------------+---------+
customer_id|customer_name|email|city|signup_date|_ingestion_timestamp|_source_file|_batch_id|
+-----------+-------------+-----+----+-----------+--------------------+------------+---------+
+-----------+-------------+-----+----+-----------+--------------------+------------+---------+

+-----------+-------------+------------------------------+---------+-----------+--------------------------+----------------------------------------------------------------+---------+
customer_id|customer_name|email |city |signup_date|_ingestion_timestamp |_source_file |_batch_id|
+-----------+-------------+------------------------------+---------+-----------+--------------------------+----------------------------------------------------------------+---------+
CUST001 |Priya Patel |priya.patel1@mailexample.in |Mumbai |2025-06-24 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST002 |Sneha Gupta |sneha.gupta2@mailexample.in |Kolkata |2025-03-13 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST003 |Divya Singh |divya.singh3@mailexample.in |Kochi |2025-02-08 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST004 |Suresh Gupta |suresh.gupta4@mailexample.in |Kochi |2023-09-03 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST005 |Aarav Gupta |aarav.gupta5@mailexample.in |Mumbai |2023-04-24 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST006 |Vikram Sharma|vikram.sharma6@mailexample.in |Kochi |2023-01-12 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST007 |Lakshmi Rao |lakshmi.rao7@mailexample.in |Pune |2023-08-04 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST009 |Anita Das |anita.das9@mailexample.in |Mumbai |2025-01-10 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST010 |Anita Sharma |anita.sharma10@mailexample.in |Hyderabad|2023-10-22 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST011 |Meera Iyer |meera.iyer11@mailexample.in |Chennai |2023-01-26 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST012 |Lakshmi Patel|lakshmi.patel12@mailexample.in|Pune |2024-04-17 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST014 |Divya Singh |divya.singh14@mailexample.in |Kochi |2025-06-07 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST015 |Priya Das |priya.das15@mailexample.in |Mumbai |2024-02-12 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST016 |Rahul Singh |rahul.singh16@mailexample.in |Chennai |2025-05-14 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST018 |Imran Reddy |imran.reddy18@mailexample.in |Bengaluru|2025-02-23 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST019 |Rohan Reddy |rohan.reddy19@mailexample.in |Mumbai |2025-01-04 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST020 |Divya Das |divya.das20@mailexample.in |Kochi |2023-05-26 |2026-11-20 09:37:55.653343|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/customers.csv|BATCH_001|
CUST021 |Meera Nair |meera.nair21@mailexample.in |Mumbai |2024-10-03 |2026